In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
import time
from selenium.webdriver.support import expected_conditions as EC


#first, open the webdriver
#cookies and login will be handled manually (as it was not possible to handle the login otherwise)
options = uc.ChromeOptions()
options.headless = False  #can set this to True if we don't want to see the code run
driver = uc.Chrome(options=options)


#we have the base URL, where all the articles will be, the news are about Brexit and its aftermath
base_url = 'https://www.thejournal.ie/brexit/news/'
all_article_urls = set()  #this is to handle duplicates in the article urls (cause i had a lot of them otherwise)

article_data = [] #create a list to store the article data
comments_data = [] #create a list to store the comments data

#for now we only loop through the first 2 pages for simplicity, but more pages will be added to cover the whole period 2016-2024 
for page_num in range(1, 3):
    #the following line constructs the base url: if it is page 1, it's just the base url, otherwise it adds a page number to the URL 
    url = base_url if page_num == 1 else f'{base_url}page/{page_num}/'
    
    #we send a GET request to that page and store the response
    response = requests.get(url)
    #we parse the HTML of the page using BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')
    #we find all anchor tags, that contain the links to individual articles, with the class 'link-overlay-redesign'
    links = soup.find_all('a', class_='link-overlay-redesign')
    #for each anchor tag found, extract the href (the actual URL of the article)
    for link in links:
        all_article_urls.add(link['href']) #this are where all the links were

#we loop through each article to scrape the data
for article_url in all_article_urls:
    #first we print the current article URL being scraped (it helps with debugging and to track the progress)
    print(f"Scraping article: {article_url}")
    
    #then we send an HTTP GET request to the web page at the URL stored, to 'open' the article 
    article_response = requests.get(article_url)
    #we parse the HTML content using BeautifulSoup
    article_soup = BeautifulSoup(article_response.text, 'html.parser')
    
    #Then: we get the article text
    #first we try to find the main content area of the article using its div class (found when inspecting the structure of the website)
    article_div = article_soup.find('div', class_='article-content-redesign')
    article_text = ""     #we iitialise a string to hold the article text
    if article_div: #if the article <div> was found 
        paragraphs = article_div.find_all('p') #we find all paragraph tags inside the article content div
        #finally we extract the text from each paragraph, strip whitespace, and join them with line breaks
        article_text = '\n'.join(p.get_text(strip=True) for p in paragraphs)
    else: #we print a warning if no article content was found
        print(f"No article text found for {article_url}")
    
    #we scrape the article creation date
    #for this we try to find the <div> tag that contains the article's creation date, using its specific class name
    creation_date_div = article_soup.find('div', class_='metadata-csp-date')
    #we initialise the variable to store the article's creation date as None (in case it's not found)
    article_date = None
    #if the <div> was found (it's not None), we can extract it
    if creation_date_div:
        #we extract the text from the <div> and remove extra whitespace
        article_date = creation_date_div.get_text(strip=True)

    #we scrape the article last updated date (if it is ever needed during the analysis)
    #for this we try to find the <span> tag that contains the article's updated time, using its specific class name
    updated_span = article_soup.find('span', class_='article-updated-time-redesign')
    #we initialise the variable to store the article's updated date as None (in case it's not found)
    article_updated_date = None
    #if the <span> was found (it's not None), we can extract it
    if updated_span:
        #we extract the text from the <span> and remove extra whitespace
        article_updated_date = updated_span.get_text(strip=True)

    #we append all the article data
    article_data.append({
        'article_url': article_url,
        'article_text': article_text,
        'article_date_created': article_date,
        'article_date_updated': article_updated_date
    })
    
    #then i need to open the article in the browser to scrape the comments
    driver.get(article_url)
    time.sleep(2)   #this is to make sure the content has time to load

    #we try to click the 'View comments' button
    #sometimes, in the structure, it's a <div> or <button> , this code handles that and will click it anyway  
    #the following 7 lines of code were found with Chatgpt 3.5
    try:
        comment_trigger = driver.find_element(By.CLASS_NAME, 'view-side-panel-comments-button')
        driver.execute_script("arguments[0].click();", comment_trigger)
        time.sleep(2)
    except Exception as e:
        print(f"Couldn't click comment section for {article_url}: {e}")
        continue  #this skip this article if no comments are found

    #now we scrape each comment with their associated date 
    comment_elements = driver.find_elements(By.CLASS_NAME, 'comment-body.text')
    date_elements = driver.find_elements(By.CLASS_NAME, 'date')

    #then we want to loop through the comment element and the date element of each comment to obtain the exact text and date
    for comment_el, date_el in zip(comment_elements, date_elements):
        #extract the text from the comment and remove the extra whitespace
        comment_text = comment_el.text.strip()
        #extract the text from the date and remove the extra whitespace
        comment_date = date_el.text.strip()
        #if the comment text is not empty, the comment data will be added to a dictionary
        if comment_text:
            comments_data.append({
                'article_url': article_url,
                'comment_text': comment_text,
                'comment_date': comment_date
            })

#finally we convert the articles and comments data into dataframes
articles_df = pd.DataFrame(article_data)
comments_df = pd.DataFrame(comments_data)



Scraping article: https://www.thejournal.ie/boris-johnson-difficult-vote-windsor-framework-6008783-Mar2023/
Scraping article: https://www.thejournal.ie/readme/housing-crisis-ireland-5-6071020-May2023/
Scraping article: https://www.thejournal.ie/protocol-talks-monday-sunak-ursula-von-der-leyen-6005263-Feb2023/
Scraping article: https://www.thejournal.ie/brexit-supports-20-would-vote-different-now-6100706-Jun2023/
Scraping article: https://www.thejournal.ie/brexit-driver-in-rise-of-english-welsh-people-holding-irish-passports-6157029-Aug2023/
Scraping article: https://www.thejournal.ie/northern-ireland-protocol-deal-dup-analysis-5997854-Feb2023/
Scraping article: https://www.thejournal.ie/progress-being-made-northern-ireland-protocol-5988752-Feb2023/
Scraping article: https://www.thejournal.ie/jeffrey-donaldson-stormont-northern-ireland-6212726-Nov2023/
Scraping article: https://www.thejournal.ie/keir-starmer-brexit-6171360-Sep2023/
Scraping article: https://www.thejournal.ie/readme/brex

In [6]:
driver.quit()

In [7]:
#and then save them to csv
articles_df.to_csv('articles_2023_2024.csv', index=False)  
comments_df.to_csv('comments_2023_2024.csv', index=False)  


In [9]:
articles_df.head()

,article_url,article_text,article_date_created,article_date_updated
0,https://www.thejournal.ie/boris-johnson-diffic...,LAST UPDATE|2 Mar 2023\nBORIS JOHNSON HAS said...,"3.44pm, 2 Mar 2023",2 Mar 2023
1,https://www.thejournal.ie/readme/housing-crisi...,BREXIT AND COVID have dominated headlines – an...,"7.01am, 22 May 2023",None
2,https://www.thejournal.ie/protocol-talks-monda...,LAST UPDATE|27 Feb 2023\n[NOTE: We are now cov...,"7.02am, 27 Feb 2023",27 Feb 2023
3,https://www.thejournal.ie/brexit-supports-20-w...,AT LEAST 20% of Brexit supporters would vote d...,"7.40am, 23 Jun 2023",None
4,https://www.thejournal.ie/brexit-driver-in-ris...,THE NUMBER OF UK-born residents in England and...,"1.34pm, 31 Aug 2023",None


In [10]:
comments_df.head()

,article_url,comment_text,comment_date
0,https://www.thejournal.ie/boris-johnson-diffic...,He should’ve sorted it while he was PM then. S...,"Mar 2nd 2023, 4:00 PM"
1,https://www.thejournal.ie/boris-johnson-diffic...,Empty vessels make plenty of noise,"Mar 2nd 2023, 4:30 PM"
2,https://www.thejournal.ie/boris-johnson-diffic...,Who cares,"Mar 2nd 2023, 3:55 PM"
3,https://www.thejournal.ie/boris-johnson-diffic...,Complete and utter w- -ker,"Mar 2nd 2023, 4:33 PM"
4,https://www.thejournal.ie/boris-johnson-diffic...,"Boris, you are as bad as the DUP! Just disappear.","Mar 2nd 2023, 4:17 PM"


In [15]:
len(articles_df)
#we currently have 80 articles

80

In [16]:
len(comments_df)
#and 1401 comments total (across the 80 articles)

1401